In [4]:
import torchvision
import torch
import numpy as np
import cv2
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pandas as pd
import torch.optim as optim
import segmentation_models_pytorch as smp
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

c:\Users\40757\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


![Mask R-CNN Architecture](./Mask_R-CNN_Arch.png)

In [5]:
# img.png, label, RLE_encoding 

# EX: 
# 0007a71bf.jpg, 3, 18661 28 18863 82 19091 110 19347 110 19603
# Starting at pixel 18661, mask lasts 28 pixels 
# Starting at pixel 18863, mask lasts 82 pixels

def RLE_to_mask(rle, shape=(256, 1600)):
    # Makes 2 lists: Mask starting pixels and Mask end pixels 
    s = rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0::2], s[1::2])]
    starts -= 1
    ends = starts + lengths

    # creates a 0-image
    img = np.zeros(shape[0]*shape[1], dtype=np.uint8)
    
    # covers the mask 
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1

    # Its F because the pixels in rle go COLUMN wise
    return img.reshape(shape, order='F')

In [ ]:
# Store dictinaries soon to be tuple of tuples (so we can shove into DataLoader)
# Tuple 0: Images (tensor_img_1, tensor_img_2)
# Tuple 1: Targets ({"boxes": tensor_boxes_1, "labels": tensor_labels_1, "masks": tensor_masks_1},
#                    {"boxes": tensor_boxes_2, "labels": tensor_labels_2, "masks": tensor_masks_2})
class SteelDataset(Dataset):
    def __init__(self, df, img_folder, img_size=(256,1600)):
        self.df = df
        self.img_folder = img_folder
        self.grouped = df.groupby('ImageId')   
        self.image_ids = list(self.grouped.groups.keys())  # list of unique img id's
        self.img_size = img_size

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        rows = self.grouped.get_group(img_id)

        image = cv2.imread(f"{self.img_folder}/{img_id}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)      # read image

        masks = []
        boxes = []
        labels = []

        # Append masks boxes and labels
        for _, row in rows.iterrows():
            mask = RLE_to_mask(row['EncodedPixels'])
            masks.append(mask)

            pos = np.where(mask)
            xmin, xmax = np.min(pos[1]), np.max(pos[1])
            ymin, ymax = np.min(pos[0]), np.max(pos[0])

            xmax = max(xmax, xmin + 1)
            ymax = max(ymax, ymin + 1)

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(row['ClassId']) 

        # Create dict
        target = {      
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "masks": torch.as_tensor(np.stack(masks), dtype=torch.uint8)
        }
        
        image = torch.from_numpy(image).permute(2,0,1).float() / 255.0

        return image, target

def collate_fn(batch):      # creates a tuple of dicts
    return tuple(zip(*batch))

In [7]:
maskRcnn_model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")

# 4 defects + background
num_classes = 5

# Box predictor
in_features = maskRcnn_model.roi_heads.box_predictor.cls_score.in_features
maskRcnn_model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

# Mask predictor
in_features_mask = maskRcnn_model.roi_heads.mask_predictor.conv5_mask.in_channels
hidden_layer = 256
maskRcnn_model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

maskRcnn_model.to(device)

MaskRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(in

In [8]:
df = pd.read_csv("../data/kaggle_data/train.csv")
df.columns = ["ImageId", "ClassId", "EncodedPixels"]

steel_data = SteelDataset(df, img_folder="../data/kaggle_data/train_images/", img_size=(256,1600))
train_loader = DataLoader(steel_data, batch_size=2, shuffle=True, collate_fn=collate_fn)

optimizer = optim.AdamW(maskRcnn_model.parameters(), lr=1e-4, weight_decay=1e-2)

def train(model, loader, optimizer, epochs=5):
    model.train()
    for epoch in range(epochs):
        total_loss = 0

        # images = Batch of images
        # targests = Batch of dict's
        for images, targets in loader:
            images = [img.to(device) for img in images]
            targets = [{k:v.to(device) for k,v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            total_loss += losses.item()

        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(loader):.4f}")

In [9]:
train(maskRcnn_model, train_loader, optimizer, epochs=10)
torch.save(maskRcnn_model.state_dict(), 'maskRcnn_model.pth')

KeyboardInterrupt: 